# Lab 15-09 · Lead and Lag Compensation

Four parts, each built on `compensate.py`. **Predict first**, then run.

In [1]:
import math
import shutil
import subprocess
import sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from compensate import lead_phase, lead_alpha, design_lead, design_lag, lead_rc, with_comp
sys.path.insert(0, "../15-07-lab"); sys.path.insert(0, "../15-03-lab")
from freqdomain import margins, freq_resp
from specs import step_curve, step_specs

## Part 1 · A lead network in ngspice

*Predict:* the maximum phase and its frequency.

In [2]:
R1, R2, C1 = 9e3, 1e3, 10e-9
alpha = (R1 + R2) / R2
fz, fp = 1 / (2 * math.pi * R1 * C1), 1 / (2 * math.pi * (R1 * R2 / (R1 + R2)) * C1)
print(f"alpha = {alpha:.0f}: predicted {lead_phase(alpha):.2f} deg at {math.sqrt(fz * fp) / 1e3:.3f} kHz")
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "lead_network.cir"], capture_output=True, text=True, timeout=120).stdout
    print("ngspice:", [l for l in out.splitlines() if l.startswith(("phmax =", "fm ="))])
else:
    print("ngspice not installed: skip")

alpha = 10: predicted 54.90 deg at 5.592 kHz
ngspice: ['phmax = 5.490320e+01', 'fm = 5.592135e+03']


## Part 2 · Lead against lag

In [3]:
G = ([10], [1, 1, 0])
lead = design_lead(*G, 45)
lag = design_lag(*G, 50)
t = np.linspace(0, 40, 40001)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
for name, d in (("lead", lead), ("lag", lag)):
    n, dd = with_comp(*G, d["cnum"], d["cden"])
    cl = np.polyadd(dd, n)
    y = step_curve(n, cl, t)
    s = step_specs(n, cl, 40)
    m = margins(n, dd)
    print(f"{name}: crossover {m['wc']:.3f} rad/s, PM {m['pm']:.1f} deg, overshoot {100 * s['overshoot']:.1f}%, settling {s['settle']:.2f} s")
    a1.plot(t, y, label=name); a2.semilogy(t, np.abs(1 - y) + 1e-6, label=name)
a1.set_xlim(0, 10); a1.set_xlabel("t (s)"); a1.legend(); a2.set_xlabel("t (s)"); a2.set_ylabel("|error|"); a2.legend()
plt.show()

lead: crossover 4.425 rad/s, PM 49.8 deg, overshoot 22.4%, settling 1.23 s


lag: crossover 0.703 rad/s, PM 49.7 deg, overshoot 22.3%, settling 19.71 s


## Part 3 · Root-locus lead: moving the zero

In [4]:
sd = complex(-2, 2 * math.sqrt(3))
for zc in (-1.5, -2.0, -2.5, -3.0):
    need = -180 + math.degrees(np.angle(sd)) + math.degrees(np.angle(sd + 1))
    angp = math.degrees(np.angle(sd - zc)) - need
    pc = sd.real - sd.imag / math.tan(math.radians(angp))
    K = abs(sd) * abs(sd + 1) * abs(sd - pc) / abs(sd - zc)
    num = np.polymul([K], [1, -zc]); den = np.polymul([1, 1, 0], [1, -pc]); cl = np.polyadd(den, num)
    third = np.roots(cl)[np.argmin(np.abs(np.roots(cl).imag))].real
    s = step_specs(num, cl, 20)
    print(f"zero {zc:5.2f}: pole {pc:6.2f}, K {K:6.2f}, third pole {third:6.2f}, overshoot {100 * s['overshoot']:5.1f}%, Kv {K * zc / pc:.2f}")

zero -1.50: pole  -4.70, K  18.09, third pole  -1.70, overshoot  23.0%, Kv 5.78


zero -2.00: pole  -5.60, K  20.80, third pole  -2.60, overshoot  27.4%, Kv 7.43


zero -2.50: pole  -6.82, K  24.47, third pole  -3.82, overshoot  29.8%, Kv 8.97


zero -3.00: pole  -8.57, K  29.71, third pole  -5.57, overshoot  30.5%, Kv 10.40


## Part 4 · A Type III compensator for an LC double pole

In [5]:
Vin, L, C, R = 5.0, 1e-6, 100e-6, 1.0
w0 = 1 / math.sqrt(L * C); wp = 2 * math.pi * 300e3; wc = 2 * math.pi * 60e3
plant = ([Vin], [L * C, L / R, 1])
from scipy.optimize import brentq
ks, pms = np.linspace(0.3, 1.5, 25), []
for k in ks:
    wz = k * w0
    def loop(wI):
        n = np.polymul(wI * np.polymul([1 / wz, 1], [1 / wz, 1]), plant[0])
        d = np.polymul(np.polymul([1, 0], np.polymul([1 / wp, 1], [1 / wp, 1])), plant[1])
        return n, d
    wI = brentq(lambda x: abs(freq_resp(*loop(x), wc)) - 1, 1, 1e9)
    pms.append(margins(*loop(wI), w_lo=1e2, w_hi=1e8)["pm"])
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(ks, pms); ax.set_xlabel("zero frequency / f0"); ax.set_ylabel("phase margin (deg)"); ax.grid(True, alpha=0.3)
plt.show()
print(f"PM at f0: {np.interp(1.0, ks, pms):.1f} deg; at f0/2: {np.interp(0.5, ks, pms):.1f} deg")

PM at f0: 39.3 deg; at f0/2: 53.9 deg


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.